# EDA


### 0. Check files and read in data
- Check that the files are uploaded
- Read in the CSV

In [0]:
# Check files

dbutils.fs.ls("/Volumes/marathos/default/raw")

In [0]:
# Read in data 

DATA_PATH = "/Volumes/marathos/default/raw"

df_raw = spark.read.csv(f"{DATA_PATH}/TWO_CENTURIES_OF_UM_RACES.csv", header=True, inferSchema=True)

### 1. Initial EDA

#### 1.1. Explore columns and schema
Find out 
- number of rows
- number of columns
- check schema if you have inferred it or specify an appropriate schema

- descriptive summary of numerical fields

In [0]:

df_raw.count()
# Find out number of rows

len(df_raw.columns)
# Find out number of columns

print(f"Number of rows:{df_raw.count()}")
print(f"Number of columns:{len(df_raw.columns)}")

In [0]:
df_raw.columns
# See all column names

df_raw.printSchema()
# See all column names and types

In [0]:
df_raw.display(10)

#### Summary of data exploration

- 7 miljon rows
- 13 columns

- Columns and Schema: 
    - Year of event: integer
    - Event dates: string -> datetieme
    - Event name: string 
    - Event distance/length: string 
    - Event number of finishers: integer 
    - Athlete performance: string -> vad ska detta vara
    - Athlete club: string 
    - Athlete country: string 
    - Athlete year of birth: double -> borde vara samma som athlete year (int)
    - Athlete gender: string
    - Athlete age category: string 
    - Athlete average speed: string -> double
    - Athlete ID: integer



#### 1.2. Explore the data

- **Nulls**
A lot of nulls for teh following categorys:
    - Age of birth
    - Age category
    - Club

    - seems to be only for 2018 - how many event where there 2018?

- **Events**
    - Count of unique events = 26.900

- how are ages distributed among the runners?
- which countries are most represented?

### Unique events

# TO DO

- kanske lista antal unika event per år?
- tydlig lista på återkommande event per år?

In [0]:
from pyspark.sql.functions import countDistinct

df_raw.select("Event name").distinct().display()
# View list of all unique events 

df_raw.groupBy("Event name") \
    .agg(countDistinct("Year of event").alias("number_of_years")) \
    .orderBy("number_of_years", ascending=False) \
    .display()
# View reaccuring events

unique_events = df_raw.select("Event name").distinct().count()
print(f"Number of unique events:{unique_events}")
# Count number of unique events



### Nulls

In [0]:
# Counts number of null values in each column
from pyspark.sql.functions import col, sum as spark_sum

null_count = df_raw.select(
    [spark_sum(col(columns).isNull().cast("int")).alias(columns) for columns in df_raw.columns
])

# Display columns with null values
null_count = null_count.collect()[0].asDict()
[(column, nulls) for column, nulls in null_count.items() if nulls > 0]

In [0]:

# Display rows with nulls for ´Athlete age category´
df_raw_nulls = df_raw.filter(col("Athlete age category").isNull())

df_raw_nulls.display()

In [0]:
# Find connectins between null values

# 1. Nulls per year
df_raw_nulls.groupBy("Year of event").count().orderBy("Year of event").display()

# 2. Nulls per event
df_raw_nulls.groupBy("Event name").count().orderBy("count", ascending=False).display()


In [0]:
# 3. Check if Athlete year of birth and Athlete age category have null on same rows

only_birth_null = df_raw.filter(
    col("Athlete year of birth").isNull() & 
    col("Athlete age category").isNotNull()
).count()

only_age_null = df_raw.filter(
    col("Athlete year of birth").isNotNull() & 
    col("Athlete age category").isNull()
).count()

both_null = df_raw.filter(
    col("Athlete year of birth").isNull() & 
    col("Athlete age category").isNull()
).count()

print(f"Only birth null: {only_birth_null}")
print(f"Only age null: {only_age_null}")
print(f"Both null: {both_null}")


In [0]:
# Look at athlete club 

df_raw_nulls_club = df_raw.filter(col("Athlete club").isNull())

df_raw_nulls_club.display()

# Find connectins between null values

# 1. Nulls per year
df_raw_nulls_club.groupBy("Year of event").count().orderBy("Year of event").display()


# 2. Nulls per event
df_raw_nulls_club.groupBy("Event name").count().orderBy("count", ascending=False).display()

# 3. Nulls per country
df_raw_nulls_club.groupBy("Athlete country").count().orderBy("count", ascending=False).display()

In [0]:
# How many % nulls for Athlete club in Two Oceans Marathon

total_two_oceans = df_raw.filter(col("Event name") == "Two Oceans Marathon (RSA)").count()
null_two_oceans = df_raw_nulls_club.filter(col("Event name") == "Two Oceans Marathon (RSA)").count()

print(f"Totalt: {total_two_oceans}")
print(f"Nulls: {null_two_oceans}")
print(f"Andel: {round(null_two_oceans / total_two_oceans * 100, 1)}%")



In [0]:
from pyspark.sql.functions import count, sum as spark_sum, round

df_raw.filter(col("Event name") == "Two Oceans Marathon (RSA)") \
    .groupBy("Year of event") \
    .agg(
        count("*").alias("total"),
        spark_sum(col("Athlete club").isNull().cast("int")).alias("nulls")
    ) \
    .withColumn("null_pct", round(col("nulls") / col("total") * 100, 1)) \
    .orderBy("Year of event") \
    .display()

In [0]:
# Look at remaning colums with null values

null_columns = [ 
    "Athlete country", 
    "Athlete gender", 
    "Athlete performance"
]

for columns in null_columns:
    df_raw.filter(col(columns).isNull()) \
        .groupBy("Event name") \
        .count() \
        .orderBy("count", ascending=False) \
        .display()

# Findings

## Null Analysis: 

#### Athlete Age Data

- `Athlete year of birth`: 588,161 nulls | `Athlete age category`: 584,938 nulls
- 584,740 rows (~8%) are missing **both** columns simultaneously → systematic missing data, not random
- Only 3,421 rows missing `year of birth` alone, and 198 missing `age category` alone
- No clear pattern linked to a specific year or event
- Age category could potentially be calculated for the 3,421 rows where `year of birth` exists, but given they represent only ~0.05% of total data the impact on any average age calculation would be negligible
- Spark's `avg()` automatically excludes nulls → ~8% of athletes excluded from any age calculations

### Athlete Club
- 2,826,373 nulls (~38% of total data)
- Two causes identified:
  - **Historical data (pre-2001):** 100% nulls — club data was not collected
  - **Modern data (2001+):** 5-23% nulls — athlete has no club affiliation, which is normal for recreational runners
- Two Oceans Marathon (RSA) accounts for 129,684 nulls — largest single contributor
- Nulls are **meaningful** and should not be dropped → recommend replacing with `"Unknown"`

### Athlete Country, Gender & Performance
- `Athlete country`: 3 nulls | `Athlete gender`: 7 nulls | `Athlete performance`: 2 nulls
- Combined, these represent 12 rows out of 7,461,195 (~0.0002%) — no clear pattern or event connection


---

## Summary — Actions Required for Cleaning

- `Athlete club`: replace nulls with `"Unknown"` — do **not** drop rows
- `Athlete year of birth` / `Athlete age category`: leave nulls as-is, Spark handles them automatically in aggregations
- `Athlete country`, `Athlete gender`, `Athlete performance`: drop the 12 affected rows — negligible impact on dataset
- No rows should be dropped based on athlete age or club nulls


